# 2. Inference по базису драйверов

Ноутбук анализирует один выбранный Markdown-кейс, записывает подробный и компактный результаты и формирует requests to add. Каталог во время inference не меняется.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%cd {ROOT}


In [ ]:
from src.config import Settings
from src.file_io import load_catalog
from src.inference import run_inference

# Укажите имя одного файла из data/test (без пути).
CASE_FILENAME = "case_082_optimal_drop_times_using_machine_learning.md"

settings = Settings.from_env()
catalog = load_catalog(settings.driver_catalog_path)
case_path = settings.test_cases_dir / CASE_FILENAME
if not case_path.is_file():
    raise FileNotFoundError(f"Test case does not exist: {case_path}")
print(f"Test case: {case_path}")
print(f"Catalog v{catalog.catalog_version}: {len(catalog.drivers)} drivers")
print(f"Model: {settings.openai_model}")


In [ ]:
results = run_inference(settings, CASE_FILENAME)
result = results[0]
print(f"Processed: {result['case_id']}")


In [ ]:
import pandas as pd
compact_path = settings.artifacts_dir / "inference" / "compact_results.csv"
pd.read_csv(compact_path)


## Полнота информации и приоритет уточнений

Полнота — взвешенная доля известной информации по применимым драйверам. Весом служит `cost_impact_percent`, а долей неизвестности — вероятность категории `unknown`. Уточнение одного драйвера снимает его взвешенную неизвестность; прирост ниже выражен в процентных пунктах общей полноты.


In [ ]:
from src.information_coverage import calculate_information_coverage
from src.schemas import DriverInference

driver_results = [DriverInference.model_validate(item) for item in result["driver_results"]]
coverage, clarification_priorities = calculate_information_coverage(driver_results, catalog)
print(f"Суммарная заполненность информации по базису: {coverage:.1f}%")


In [ ]:
priority_table = pd.DataFrame(clarification_priorities).rename(columns={
    "driver_id": "Драйвер",
    "driver_name": "Название",
    "cost_impact_percent": "Влияние на стоимость, %",
    "unknown_probability_percent": "Неизвестность, %",
    "coverage_gain_percentage_points": "Прирост полноты, п.п.",
    "removable_uncertainty_percent": "Снимаемая неопределённость, %",
})
priority_table = priority_table[priority_table["Прирост полноты, п.п."] > 0].reset_index(drop=True)
priority_table.index += 1
priority_table.round(2)
